In [3]:
# =========================================================
# DAILY MAX HEAT INDEX STACKS
# QUARTERLY MULTIBAND TIFFS
#
# 2023 + 2024
#
# BAND = DAILY MAX HI
# (11 AM–5 PM IST)
#
# OUTPUT:
#
# HI_2023_Q1.tif
# HI_2023_Q2.tif
# HI_2023_Q3.tif
# HI_2023_Q4.tif
#
# HI_2024_Q1.tif
# HI_2024_Q2.tif
# HI_2024_Q3.tif
# HI_2024_Q4.tif
#
# GRID MATCHES:
# JFM_P80_1990_2023.tif
# =========================================================

import ee
import geemap
import geopandas as gpd
import rasterio
from pathlib import Path

# =========================================================
# INITIALIZE
# =========================================================

ee.Authenticate()
ee.Initialize(project="areca-farm")

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-assam/data_extractor/era5_land"
)

geojson_path = ROOT.parent / "Maps" / "Geojson" / "assam_rc_2025-04.geojson"

# spatial template for export
percentile_tif = (
    ROOT
    / "data"
    / "percentile_tiffs"
    / "JFM_P80_1990_2023.tif"
)

# =========================================================
# READ TARGET GRID
# =========================================================

with rasterio.open(percentile_tif) as src:

    transform = src.transform

    print("\nTarget Grid")
    print("Shape:", src.shape)
    print("Resolution:", src.res)
    print("Transform:", transform)

# Earth Engine CRS Transform

crs_transform = [
    transform.a,
    transform.b,
    transform.c,
    transform.d,
    transform.e,
    transform.f
]

print("\nUsing CRS Transform:")
print(crs_transform)

# =========================================================
# LOAD DISTRICTS
# =========================================================

gdf = gpd.read_file(geojson_path)

gdf["geometry"] = gdf.geometry.simplify(0.01)

districts = geemap.geopandas_to_ee(gdf)

region = districts.geometry()

# =========================================================
# ERA5 LAND HOURLY
# =========================================================

era5 = ee.ImageCollection(
    "ECMWF/ERA5_LAND/HOURLY"
)

# =========================================================
# IST WINDOW
#
# 11:00–17:00 IST
# = 06:00–12:00 UTC
# =========================================================

utc_start = 6
utc_end = 12

# =========================================================
# HEAT INDEX
# =========================================================

def compute_hi(image):

    T = (
        image
        .select("temperature_2m")
        .subtract(273.15)
        .rename("T")
    )

    Td = (
        image
        .select("dewpoint_temperature_2m")
        .subtract(273.15)
        .rename("Td")
    )

    RH = Td.expression(
        """
        100 * (
            exp((17.625 * Td)/(243.04 + Td))
            /
            exp((17.625 * T)/(243.04 + T))
        )
        """,
        {
            "Td": Td,
            "T": T
        }
    )

    HI = T.expression(
        """
        -8.784695 +
        1.61139411*T +
        2.338549*RH -
        0.14611605*T*RH -
        0.012308094*T*T -
        0.016424828*RH*RH +
        0.002211732*T*T*RH +
        0.00072546*T*RH*RH -
        0.000003582*T*T*RH*RH
        """,
        {
            "T": T,
            "RH": RH
        }
    )

    return HI.rename("HI")

# =========================================================
# DAILY MAX HI
# =========================================================

def daily_max_hi(day):

    day = ee.Date(day)

    daily = (
        era5
        .filterDate(
            day,
            day.advance(1, "day")
        )
        .filter(
            ee.Filter.calendarRange(
                utc_start,
                utc_end,
                "hour"
            )
        )
    )

    # Add temperature band
    daily = daily.map(
        lambda img: img.addBands(
            img.select("temperature_2m")
            .subtract(273.15)
            .rename("T")
        )
    )

    # Hottest hour
    hottest = daily.qualityMosaic("T")

    hi = compute_hi(hottest)

    return (
        hi
        .rename(
            day.format("YYYYMMdd")
        )
        .set(
            "system:time_start",
            day.millis()
        )
    )

# =========================================================
# QUARTERS
# =========================================================

quarters = {
    "Q1": ("2023-01-01", "2023-04-01"),
    "Q2": ("2023-04-01", "2023-07-01"),
    "Q3": ("2023-07-01", "2023-10-01"),
    "Q4": ("2023-10-01", "2024-01-01")
}

quarters_2024 = {
    "Q1": ("2024-01-01", "2024-04-01"),
    "Q2": ("2024-04-01", "2024-07-01"),
    "Q3": ("2024-07-01", "2024-10-01"),
    "Q4": ("2024-10-01", "2025-01-01")
}

# =========================================================
# EXPORT FUNCTION
# =========================================================

def export_quarter(year, quarter, start, end):

    print(
        f"\nPreparing {year} {quarter}"
    )

    start_date = ee.Date(start)
    end_date = ee.Date(end)

    ndays = end_date.difference(
        start_date,
        "day"
    )

    days = ee.List.sequence(
        0,
        ndays.subtract(1)
    )

    daily_images = ee.ImageCollection.fromImages(

        days.map(
            lambda d:
            daily_max_hi(
                start_date.advance(
                    d,
                    "day"
                )
            )
        )

    )

    stack = daily_images.toBands()

    print(
        "Bands:",
        daily_images.size().getInfo()
    )

    task = ee.batch.Export.image.toDrive(

        image=stack.clip(region),

        description=f"HI_{year}_{quarter}",

        folder="Daily_Heat_Index_Assam",

        fileNamePrefix=f"HI_{year}_{quarter}",

        region=region,

        crs="EPSG:4326",

        crsTransform=crs_transform,

        maxPixels=1e13
    )

    task.start()

    print(
        f"Submitted HI_{year}_{quarter}"
    )

# =========================================================
# 2023
# =========================================================

for q, (s, e) in quarters.items():

    export_quarter(
        2023,
        q,
        s,
        e
    )

# =========================================================
# 2024
# =========================================================

for q, (s, e) in quarters_2024.items():

    export_quarter(
        2024,
        q,
        s,
        e
    )

print("\n===================================")
print("ALL 8 EXPORTS SUBMITTED")
print("===================================")


Target Grid
Shape: (48, 79)
Resolution: (0.08084837557075693, 0.08084837557075693)
Transform: | 0.08, 0.00, 89.66|
| 0.00,-0.08, 27.97|
| 0.00, 0.00, 1.00|

Using CRS Transform:
[0.08084837557075693, 0.0, 89.66084850796943, 0.0, -0.08084837557075693, 27.973537947481898]

Preparing 2023 Q1
Bands: 90
Submitted HI_2023_Q1

Preparing 2023 Q2
Bands: 91
Submitted HI_2023_Q2

Preparing 2023 Q3
Bands: 92
Submitted HI_2023_Q3

Preparing 2023 Q4
Bands: 92
Submitted HI_2023_Q4

Preparing 2024 Q1
Bands: 91
Submitted HI_2024_Q1

Preparing 2024 Q2
Bands: 91
Submitted HI_2024_Q2

Preparing 2024 Q3
Bands: 92
Submitted HI_2024_Q3

Preparing 2024 Q4
Bands: 92
Submitted HI_2024_Q4

ALL 8 EXPORTS SUBMITTED


In [ ]:
# last 5 years summer months

In [5]:
# =========================================================
# DAILY MAX HEAT INDEX STACKS
# Q2 ONLY (APR-JUN)
#
# YEARS:
# 2021
# 2022
# 2023
# 2024
# 2025
#
# OUTPUT:
#
# HI_2021_Q2.tif
# HI_2022_Q2.tif
# HI_2023_Q2.tif
# HI_2024_Q2.tif
# HI_2025_Q2.tif
#
# BAND = DAILY MAX HI
# (11 AM–5 PM IST)
#
# GRID MATCHES:
# JFM_P80_1990_2023.tif
# =========================================================

import ee
import geemap
import geopandas as gpd
import rasterio
from pathlib import Path

# =========================================================
# INITIALIZE
# =========================================================

ee.Authenticate()
ee.Initialize(project="odishaextreme-heat")

# =========================================================
# PATHS
# =========================================================

ROOT = Path(
    "/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/era5_land"
)

geojson_path = (
    ROOT.parent
    / "assets"
    / "district.geojson"
)

percentile_tif = (
    ROOT
    / "data"
    / "percentile_tiffs"
    / "JFM_P80_1990_2023.tif"
)

# =========================================================
# READ TARGET GRID
# =========================================================

with rasterio.open(percentile_tif) as src:

    transform = src.transform

    print("\nTarget Grid")
    print("Shape:", src.shape)
    print("Resolution:", src.res)

crs_transform = [
    transform.a,
    transform.b,
    transform.c,
    transform.d,
    transform.e,
    transform.f
]

print("\nCRS Transform:")
print(crs_transform)

# =========================================================
# LOAD DISTRICTS
# =========================================================

gdf = gpd.read_file(geojson_path)

gdf["geometry"] = gdf.geometry.simplify(0.01)

districts = geemap.geopandas_to_ee(gdf)

region = districts.geometry()

# =========================================================
# ERA5 LAND
# =========================================================

era5 = ee.ImageCollection(
    "ECMWF/ERA5_LAND/HOURLY"
)

# =========================================================
# IST WINDOW
#
# 11:00–17:00 IST
# = 06:00–12:00 UTC
# =========================================================

utc_start = 6
utc_end = 12

# =========================================================
# HEAT INDEX FUNCTION
# =========================================================

def compute_hi(image):

    T = (
        image.select("temperature_2m")
        .subtract(273.15)
        .rename("T")
    )

    Td = (
        image.select("dewpoint_temperature_2m")
        .subtract(273.15)
        .rename("Td")
    )

    RH = Td.expression(
        """
        100 * (
            exp((17.625 * Td)/(243.04 + Td))
            /
            exp((17.625 * T)/(243.04 + T))
        )
        """,
        {
            "Td": Td,
            "T": T
        }
    )

    HI = T.expression(
        """
        -8.784695 +
        1.61139411*T +
        2.338549*RH -
        0.14611605*T*RH -
        0.012308094*T*T -
        0.016424828*RH*RH +
        0.002211732*T*T*RH +
        0.00072546*T*RH*RH -
        0.000003582*T*T*RH*RH
        """,
        {
            "T": T,
            "RH": RH
        }
    )

    return HI.rename("HI")

# =========================================================
# DAILY MAX HI
# =========================================================

def daily_max_hi(day):

    day = ee.Date(day)

    daily = (
        era5
        .filterDate(
            day,
            day.advance(1, "day")
        )
        .filter(
            ee.Filter.calendarRange(
                utc_start,
                utc_end,
                "hour"
            )
        )
    )

    daily = daily.map(
        lambda img: img.addBands(
            img.select("temperature_2m")
            .subtract(273.15)
            .rename("T")
        )
    )

    hottest = daily.qualityMosaic("T")

    hi = compute_hi(hottest)

    return (
        hi.rename(day.format("YYYYMMdd"))
        .set(
            "system:time_start",
            day.millis()
        )
    )

# =========================================================
# EXPORT FUNCTION
# =========================================================

def export_q2(year):

    start = f"{year}-04-01"
    end   = f"{year}-07-01"

    print(f"\nPreparing {year} Q2")

    start_date = ee.Date(start)
    end_date = ee.Date(end)

    ndays = end_date.difference(
        start_date,
        "day"
    )

    days = ee.List.sequence(
        0,
        ndays.subtract(1)
    )

    daily_images = ee.ImageCollection.fromImages(

        days.map(
            lambda d:
            daily_max_hi(
                start_date.advance(
                    d,
                    "day"
                )
            )
        )

    )

    stack = daily_images.toBands()

    print(
        "Bands:",
        daily_images.size().getInfo()
    )

    task = ee.batch.Export.image.toDrive(

        image=stack.clip(region),

        description=f"HI_{year}_Q2",

        folder="Heat_Index",

        fileNamePrefix=f"HI_{year}_Q2",

        region=region,

        crs="EPSG:4326",

        crsTransform=crs_transform,

        maxPixels=1e13
    )

    task.start()

    print(
        f"Submitted HI_{year}_Q2"
    )

# =========================================================
# EXPORT 2021-2025
# =========================================================

for year in range(2021, 2026):

    export_q2(year)

print("\n===================================")
print("ALL Q2 EXPORTS SUBMITTED")
print("2021-2025")
print("===================================")


Target Grid
Shape: (60, 77)
Resolution: (0.08084837557075693, 0.08084837557075693)

CRS Transform:
[0.08084837557075693, 0.0, 81.33346582418147, 0.0, -0.08084837557075693, 22.63754515981194]

Preparing 2021 Q2
Bands: 91
Submitted HI_2021_Q2

Preparing 2022 Q2
Bands: 91
Submitted HI_2022_Q2

Preparing 2023 Q2
Bands: 91
Submitted HI_2023_Q2

Preparing 2024 Q2
Bands: 91
Submitted HI_2024_Q2

Preparing 2025 Q2
Bands: 91
Submitted HI_2025_Q2

ALL Q2 EXPORTS SUBMITTED
2021-2025


In [3]:
import ee

ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="odishaextreme-heat")

print(ee.Number(1).getInfo())

1
